# 05 · Orquestación con un Job de Databricks

**Objetivo:** que el pipeline corra **solo, todos los días**, con dependencias, parámetros, reintentos, horario y alertas.

    ingesta ──► transformacion ──► validacion ──► publicacion
    API→RAW→bronze   bronze→silver    reglas de calidad    silver→gold + log

| Capacidad | Cómo se configura |
|---|---|
| Dependencias | `depends_on`: cada tarea espera a la anterior |
| Parámetros | `dias`, `min_filas`, etc., sin tocar el código |
| Reintentos | `ingesta` reintenta 1 vez (la API puede fallar) |
| Horario | Todos los días a las 08:00 (hora de Lima) |
| Alerta | Correo si el Job falla |
| Log | Tabla `pipeline_log` con una fila por corrida |

In [0]:
%run ./00_Setup

In [0]:
PROYECTO_DIR = os.getcwd()          # carpeta del proyecto en el Workspace
JOB_NAME     = "proyecto_opensky_brasil"
print(PROYECTO_DIR)
print(sorted(os.listdir(f"{PROYECTO_DIR}/tareas")))

## Paso 1 · Definir el Job
`dias = 3`: cada corrida revisa los **últimos 3 días**. Lo ya descargado se salta (idempotencia), así que si un día
falló por el límite de la API, la corrida siguiente lo completa sola.

In [0]:
job = {
    "name": JOB_NAME,
    "parameters": [                                                  # PARÁMETROS por ejecución
        {"name": "dias",        "default": "3"},
        {"name": "start_date",  "default": ""},
        {"name": "aeropuertos", "default": ""},
        {"name": "min_filas",   "default": "1000"},
    ],
    "tasks": [
        {"task_key": "ingesta",
         "notebook_task": {"notebook_path": f"{PROYECTO_DIR}/tareas/1_ingesta"},
         "max_retries": 1, "min_retry_interval_millis": 60000},      # REINTENTO: la API puede fallar
        {"task_key": "transformacion",
         "depends_on": [{"task_key": "ingesta"}],                    # DEPENDENCIA
         "notebook_task": {"notebook_path": f"{PROYECTO_DIR}/tareas/2_transformacion"}},
        {"task_key": "validacion",
         "depends_on": [{"task_key": "transformacion"}],
         "notebook_task": {"notebook_path": f"{PROYECTO_DIR}/tareas/3_validacion"},
         "disable_auto_optimization": True},                         # sin reintentos: repetir no arregla datos malos
        {"task_key": "publicacion",
         "depends_on": [{"task_key": "validacion"}],
         "notebook_task": {"notebook_path": f"{PROYECTO_DIR}/tareas/4_publicacion"}},
        {"task_key": "mantenimiento",
         "depends_on": [{"task_key": "publicacion"}],
         "notebook_task": {"notebook_path": f"{PROYECTO_DIR}/tareas/5_mantenimiento"}},
    ],
    "schedule": {                                                    # HORARIO: todos los días 08:00 Lima
        "quartz_cron_expression": "0 0 8 * * ?",
        "timezone_id": "America/Lima",
        "pause_status": "UNPAUSED",
    },
    "email_notifications": {"on_failure": [USER]},                   # ALERTA por correo
}
print("Job definido:", job["name"], "| tareas:", [t["task_key"] for t in job["tasks"]])

In [0]:
JOB_ID = create_job(job)

👀 **Abre el enlace:** verás las cuatro tareas encadenadas y, a la derecha, el horario, los parámetros y las notificaciones.

## Paso 2 · Ejecutarlo ahora

In [0]:
run_job(JOB_ID, dias="3")

In [0]:
display(spark.table(LOG).orderBy(F.col("ejecutado_en").desc()))

👀 **Qué observar:** apareció una nueva fila en `pipeline_log`, escrita por el Job (detalle = "Pipeline completo (Job)").

## Paso 3 · Una validación que falla
Pedimos un mínimo de vuelos imposible. La publicación **no debe correr**: gold queda intacto.

In [0]:
filas_antes = spark.table(FACT).count()
run_job(JOB_ID, min_filas="100000000")
print("Vuelos en fact antes:", filas_antes, "| después:", spark.table(FACT).count())

👀 **Qué observar:** `validacion` = FAILED y `publicacion` = UPSTREAM_FAILED. Gold no cambió y llegó un correo de alerta.

## ✔ Checkpoint
- [ ] El Job tiene 4 tareas encadenadas
- [ ] Está programado todos los días a las 08:00 (UNPAUSED)
- [ ] Una corrida exitosa dejó una fila en `pipeline_log`
- [ ] Una validación fallida detuvo la publicación

**Siguiente:** Power BI